# Chapter 37: Edge AI & Model Optimization
**Artificial Intelligence Learning Hub — Level 4**
*Pengembang: Aji Pamungkas Tri Nurcahyo — Institut Teknologi Sumatera*

Notebook pendamping ini mempraktikkan optimasi model AI untuk inferensi tepi (*on-device CPU inference*):
1. **Baseline Model Profiling:** Jumlah parameter (`45,764`), ukuran `.pt` (`183.94 KB`), akurasi (`92.33%`), dan latensi.
2. **Quantization Math:** Perhitungan manual *Symmetric* vs *Asymmetric Affine* FP32 $\to$ INT8.
3. **PyTorch Dynamic & Static PTQ:** Konversi `onednn`, kalibrasi 250 sampel, dan audit akurasi/ukuran.
4. **Weight Pruning:** *Unstructured L1* (realitas penyimpanan padat vs gzip) vs *Structured Neuron Pruning*.
5. **Knowledge Distillation:** Teacher (45.7K) membimbing Student (1.14K, **8.82 KB** = **20.85x lebih kecil**) dengan $T=3.0$.
6. **Hardware Constraints Assessment:** Evaluasi anggaran simulasi SRAM/Flash mikrokontroler.


In [2]:
import sys, os, warnings
warnings.filterwarnings("ignore")
import torch, numpy as np

sys.path.append(os.path.abspath("../code"))

from edge_ai_foundations import train_baseline_model, evaluate_accuracy
from model_profiling import get_serialized_size_kb, profile_cpu_latency
from fp32_int8_quantization_math import symmetric_quantize_int8, asymmetric_quantize_int8, dequantize_int8
from dynamic_quantization import apply_dynamic_quantization
from static_quantization_demo import build_static_quantized_model
from structured_pruning_demo import apply_structured_l1_pruning
from knowledge_distillation import train_student_distilled

print(f"PyTorch: {torch.__version__} | Quantized Engine: {torch.backends.quantized.engine}")


PyTorch: 2.15.0+cpu | Quantized Engine: onednn


## 1. Baseline FP32 Model Profiling
Melatih `EdgeTeacherMLP (16 -> 256 -> 128 -> 64 -> 4)` pada dataset sensor IoT 4-kelas (*seed=42*) dan mengukur profil dasar.


In [4]:
fp32_model, (X_train, y_train, X_test, y_test) = train_baseline_model(epochs=25, seed=42)
base_params = sum(p.numel() for p in fp32_model.parameters())
base_kb = get_serialized_size_kb(fp32_model)
base_acc = evaluate_accuracy(fp32_model, X_test, y_test) * 100
base_lat = profile_cpu_latency(fp32_model, X_test[:1], warmup=50, n_runs=500)

print(f"Params: {base_params:,} | Raw .pt Size: {base_kb:.2f} KB | Test Acc: {base_acc:.2f}%")
print(f"Mean Latency: {base_lat['mean_ms']:.4f} ms | P95: {base_lat['p95_ms']:.4f} ms")


Params: 45,764 | Raw .pt Size: 183.94 KB | Test Acc: 92.33%
Mean Latency: 0.0779 ms | P95: 0.1025 ms


## 2. Matematika Kuantisasi FP32 $\to$ INT8
Membandingkan pemetaan *Symmetric* ($z=0$) vs *Asymmetric Affine* ($z \neq 0$) pada tensor miring `[-0.50, 0.00, 0.25, 1.20, 2.54, 4.00]`.


In [6]:
x = np.array([-0.50, 0.00, 0.25, 1.20, 2.54, 4.00], dtype=np.float32)
q_sym, s_sym, z_sym = symmetric_quantize_int8(x)
err_sym = float(np.mean((x - dequantize_int8(q_sym, s_sym, z_sym)) ** 2))

q_asym, s_asym, z_asym = asymmetric_quantize_int8(x)
err_asym = float(np.mean((x - dequantize_int8(q_asym, s_asym, z_asym)) ** 2))

print(f"Symmetric  (s={s_sym:.5f}, z={z_sym}): q={q_sym.tolist()} | MSE={err_sym:.3e}")
print(f"Asymmetric (s={s_asym:.5f}, z={z_asym}): q={q_asym.tolist()} | MSE={err_asym:.3e}")


Symmetric  (s=0.03150, z=0): q=[-16, 0, 8, 38, 81, 127] | MSE=6.923e-05
Asymmetric (s=0.01765, z=-100): q=[-128, -100, -86, -32, 44, 127] | MSE=1.522e-05


## 3. PyTorch Dynamic & Static Quantization (PTQ)
Menerapkan `quantize_dynamic` dan kalibrasi *Static Quantization* pada 250 sampel pelatihan.


In [8]:
dyn_q = apply_dynamic_quantization(fp32_model)
stat_q = build_static_quantized_model(fp32_model, X_train[:250])

print(f"Dynamic INT8 : Size={get_serialized_size_kb(dyn_q):.2f} KB ({183.94/52.63:.2f}x smaller) | Acc={evaluate_accuracy(dyn_q, X_test, y_test)*100:.2f}%")
print(f"Static INT8  : Size={get_serialized_size_kb(stat_q):.2f} KB ({183.94/62.70:.2f}x smaller) | Acc={evaluate_accuracy(stat_q, X_test, y_test)*100:.2f}%")


Dynamic INT8 : Size=52.63 KB (3.49x smaller) | Acc=92.17%
Static INT8  : Size=62.70 KB (2.93x smaller) | Acc=92.67%


## 4. Structured Pruning & Knowledge Distillation
Menguji pemangkasan neuron terstruktur (50% neuron tersembunyi) serta distilasi guru–murid ($T=3.0, \alpha=0.6$).


In [10]:
struct_m = apply_structured_l1_pruning(fp32_model, keep_ratio=0.50)
distilled_m = train_student_distilled(fp32_model, X_train, y_train, epochs=25, temperature=3.0, alpha=0.6, seed=42)

print(f"Structured Pruned (0-shot) : Params={sum(p.numel() for p in struct_m.parameters()):,} | Size={get_serialized_size_kb(struct_m):.2f} KB")
print(f"Distilled Student          : Params={sum(p.numel() for p in distilled_m.parameters()):,} | Size={get_serialized_size_kb(distilled_m):.2f} KB | Acc={evaluate_accuracy(distilled_m, X_test, y_test)*100:.2f}%")


Structured Pruned (0-shot) : Params=12,644 | Size=54.57 KB
Distilled Student          : Params=1,140 | Size=8.82 KB | Acc=91.33%


## 5. Kesimpulan Trade-Off Edge AI
1. **Dynamic INT8 (`52.63 KB`) & Static INT8 (`62.70 KB`)** memangkas ukuran berkas ~3–3.5x sambil mempertahankan akurasi tinggi (`92.17%–92.67%`), namun memiliki overhead konversi kernel di CPU Windows x86 untuk tensor kecil batch=1 (`1.09–2.02 ms`).
2. **Distilled Student (`8.82 KB`, `1,140 param`)** mencapai rasio kompresi ruang **`20.85x`** sekaligus mempercepat inferensi CPU menjadi **`0.0503 ms`** dengan akurasi **`91.33%`**.
3. **Keterbatasan Mikrokontroler:** Pengujian CPU lokal tidak menggantikan verifikasi pada silikon fisik (ARM Cortex-M7 / ESP32-S3).
